In [1]:
# Imports and device setup
import os
import glob
import numpy as np
import pandas as pd
from typing import List, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score

# Optional: install torchdiffeq if not present
try:
    from torchdiffeq import odeint
except ImportError:
    %pip install -q torchdiffeq
    from torchdiffeq import odeint

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


Note: you may need to restart the kernel to use updated packages.
Using device: cpu


In [2]:
!pip install torchdiffeq

In [3]:
# =========================
# 1. LOAD + IMPUTE + NORMALIZE
# =========================

RAW_CSV_PATH = "/kaggle/input/datasets/salikhussaini49/prediction-of-sepsis/Dataset.csv"  # adjust if needed
PROCESSED_CSV_PATH = "./prediction_of_sepsis_processed.csv"

PATIENT_COL = "Patient_ID"   # change if your ID col differs
TIME_COL    = "Hour"         # change if your time col differs
LABEL_COL   = "SepsisLabel"  # change if your label col differs

# Load raw
df_raw = pd.read_csv(RAW_CSV_PATH)
print("Raw columns:", df_raw.columns.tolist())
print("Shape:", df_raw.shape)

# Identify feature columns (all numeric except id/time/label)
ignore_cols = {PATIENT_COL, TIME_COL, LABEL_COL}
feature_cols = [
    c for c in df_raw.columns
    if c not in ignore_cols and np.issubdtype(df_raw[c].dtype, np.number)
]

print("Feature columns:", feature_cols)
print("Total features:", len(feature_cols))

# Sort by patient + time
df = df_raw.sort_values([PATIENT_COL, TIME_COL]).reset_index(drop=True)

# --- Per-patient imputation: forward-fill then backward-fill ---
def impute_patient(group: pd.DataFrame) -> pd.DataFrame:
    g = group.copy()
    g[feature_cols] = g[feature_cols].ffill().bfill()
    return g

df = df.groupby(PATIENT_COL, group_keys=False).apply(impute_patient)

# Global mean imputation for any remaining NaNs
global_means = df[feature_cols].mean()
df[feature_cols] = df[feature_cols].fillna(global_means)

# --- Normalize features (global mean/std) ---
means = df[feature_cols].mean()
stds = df[feature_cols].std().replace(0, 1.0)

df[feature_cols] = (df[feature_cols] - means) / stds

print("Any remaining NaNs?", df[feature_cols].isna().any().any())

# Save processed CSV
df.to_csv(PROCESSED_CSV_PATH, index=False)
print("Saved processed data to:", PROCESSED_CSV_PATH)


Raw columns: ['Unnamed: 0', 'Hour', 'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2', 'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine', 'Bilirubin_direct', 'Glucose', 'Lactate', 'Magnesium', 'Phosphate', 'Potassium', 'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT', 'WBC', 'Fibrinogen', 'Platelets', 'Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime', 'ICULOS', 'SepsisLabel', 'Patient_ID']
Shape: (1552210, 44)
Feature columns: ['Unnamed: 0', 'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2', 'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine', 'Bilirubin_direct', 'Glucose', 'Lactate', 'Magnesium', 'Phosphate', 'Potassium', 'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT', 'WBC', 'Fibrinogen', 'Platelets', 'Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime', 'ICULOS']
Total features: 41


/tmp/ipykernel_57/1895471061.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(PATIENT_COL, group_keys=False).apply(impute_patient)


Any remaining NaNs? False
Saved processed data to: ./prediction_of_sepsis_processed.csv


In [4]:
# =========================
# 2. PATIENT-WISE TRAIN/VAL SPLIT + DATASET
# =========================

class SepsisWindowDataset(Dataset):
    """Builds fixed-length windows from a processed sepsis DataFrame.
    Each sample: (times [L], features [L,D], label [scalar]).
    Label is 1 if sepsis occurs within the next `horizon` hours after the window.
    """
    def __init__(self, df: pd.DataFrame, window_size: int = 24, horizon: int = 6, min_hours: int = 24):
        super().__init__()
        self.window_size = window_size
        self.horizon = horizon

        # Ensure sorted
        df = df.sort_values([PATIENT_COL, TIME_COL]).reset_index(drop=True)

        self.samples: List[Tuple[np.ndarray, np.ndarray, int]] = []

        for pid, g in df.groupby(PATIENT_COL):
            g = g.reset_index(drop=True)

            if len(g) < max(min_hours, window_size + horizon):
                continue

            times = g[TIME_COL].values.astype(np.float32)
            feats = g[feature_cols].values.astype(np.float32)
            labels = g[LABEL_COL].values.astype(np.float32)

            max_start = len(g) - (window_size + horizon)
            for start in range(max_start + 1):
                end = start + window_size

                # future labels within horizon
                future_labels = labels[end : end + self.horizon]
                if future_labels.size == 0:
                    continue
                y_target = future_labels.max()  # any sepsis in horizon

                t_window = times[start:end]
                x_window = feats[start:end]

                self.samples.append((t_window, x_window, int(y_target)))

        print(f"[Dataset] samples: {len(self.samples)} | features: {len(feature_cols)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        t, x, y = self.samples[idx]
        # normalize time to start at 0
        t = t - t[0]
        return (
            torch.from_numpy(t),        # [L]
            torch.from_numpy(x),        # [L, D]
            torch.tensor(y, dtype=torch.float32),
        )


def collate_pad(batch):
    ts, xs, ys = zip(*batch)
    t = torch.stack(ts, dim=0)
    x = torch.stack(xs, dim=0)
    y = torch.stack(ys, dim=0)
    return t.to(DEVICE), x.to(DEVICE), y.to(DEVICE)


# Load processed data
print("Loading processed data for splitting...")
df_processed = pd.read_csv(PROCESSED_CSV_PATH)

# Patient-wise split
all_pids = df_processed[PATIENT_COL].unique()
np.random.shuffle(all_pids)

split_idx = int(0.8 * len(all_pids))
train_pids = set(all_pids[:split_idx])
val_pids   = set(all_pids[split_idx:])

df_train = df_processed[df_processed[PATIENT_COL].isin(train_pids)].reset_index(drop=True)
df_val   = df_processed[df_processed[PATIENT_COL].isin(val_pids)].reset_index(drop=True)

print("Train patients:", len(train_pids), "| Val patients:", len(val_pids))

window_size = 24
horizon = 6
batch_size = 64

train_ds = SepsisWindowDataset(df_train, window_size=window_size, horizon=horizon, min_hours=24)
val_ds   = SepsisWindowDataset(df_val,   window_size=window_size, horizon=horizon, min_hours=24)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_pad)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, collate_fn=collate_pad)

print("Train batches:", len(train_loader), "| Val batches:", len(val_loader))


Loading processed data for splitting...
Train patients: 32268 | Val patients: 8068
[Dataset] samples: 403913 | features: 41
[Dataset] samples: 99077 | features: 41
Train batches: 6312 | Val batches: 1549


In [5]:
# =========================
# 3. TRAIN / EVAL HELPERS
# =========================

import torch.nn.functional as F


def train_one_epoch(model, loader, optimizer, epoch: int, prefix: str = "Model"):
    model.train()
    total_loss = 0.0
    total = 0
    for t, x, y in loader:
        optimizer.zero_grad()
        logits = model(t, x)
        loss = F.binary_cross_entropy_with_logits(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * y.size(0)
        total += y.size(0)
    print(f"[{prefix}] Epoch {epoch} train loss = {total_loss/total:.4f}")


@torch.no_grad()
def evaluate(model, loader, prefix: str = "Model"):
    model.eval()
    total_loss = 0.0
    total = 0
    all_logits = []
    all_labels = []
    for t, x, y in loader:
        logits = model(t, x)
        loss = F.binary_cross_entropy_with_logits(logits, y)
        total_loss += loss.item() * y.size(0)
        total += y.size(0)
        all_logits.append(logits.cpu())
        all_labels.append(y.cpu())
    avg_loss = total_loss / total
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    probs = torch.sigmoid(all_logits).numpy()
    labels_np = all_labels.numpy()
    auroc = roc_auc_score(labels_np, probs)
    print(f"[{prefix}] Val loss = {avg_loss:.4f} | AUROC = {auroc:.4f}")
    return avg_loss, auroc


In [6]:
# =========================
# 4. NEURAL ODE MODEL + TRAINING
# =========================

class ODEFunc(nn.Module):
    def __init__(self, latent_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.Softplus(),
            nn.Linear(64, 64),
            nn.Softplus(),
            nn.Linear(64, latent_dim),
        )
        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, t, z):
        return self.net(z)


class NeuralODESepsis(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int = 32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim),
        )
        self.odefunc = ODEFunc(latent_dim)
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, t: torch.Tensor, x: torch.Tensor):
        B, L, D = x.shape
        x_last = x[:, -1, :]
        z0 = self.encoder(x_last)
        # integrate over a scaled time interval [0, 1]
        t_eval = torch.tensor([0.0, 1.0], device=t.device)
        z_traj = odeint(self.odefunc, z0, t_eval)
        zT = z_traj[-1]
        logits = self.classifier(zT).squeeze(-1)
        return logits


input_dim = len(feature_cols)
model_ode = NeuralODESepsis(input_dim=input_dim, latent_dim=32).to(DEVICE)
optimizer_ode = torch.optim.Adam(model_ode.parameters(), lr=1e-3)

epochs = 10
for epoch in range(1, epochs + 1):
    train_one_epoch(model_ode, train_loader, optimizer_ode, epoch, prefix="NeuralODE")
    evaluate(model_ode, val_loader, prefix="NeuralODE")


[NeuralODE] Epoch 1 train loss = 0.1143
[NeuralODE] Val loss = 0.1117 | AUROC = 0.8053
[NeuralODE] Epoch 2 train loss = 0.1055
[NeuralODE] Val loss = 0.1171 | AUROC = 0.7983
[NeuralODE] Epoch 3 train loss = 0.0997
[NeuralODE] Val loss = 0.1261 | AUROC = 0.7809
[NeuralODE] Epoch 4 train loss = 0.0952
[NeuralODE] Val loss = 0.1403 | AUROC = 0.7618
[NeuralODE] Epoch 5 train loss = 0.0909
[NeuralODE] Val loss = 0.1473 | AUROC = 0.7628
[NeuralODE] Epoch 6 train loss = 0.0872
[NeuralODE] Val loss = 0.1534 | AUROC = 0.7538
[NeuralODE] Epoch 7 train loss = 0.0841
[NeuralODE] Val loss = 0.1681 | AUROC = 0.7455
[NeuralODE] Epoch 8 train loss = 0.0810
[NeuralODE] Val loss = 0.1805 | AUROC = 0.7476
[NeuralODE] Epoch 9 train loss = 0.0774
[NeuralODE] Val loss = 0.1763 | AUROC = 0.7253
[NeuralODE] Epoch 10 train loss = 0.0748
[NeuralODE] Val loss = 0.1944 | AUROC = 0.7448


In [7]:
# =========================
# 5. LSTM BASELINE MODEL + TRAINING
# =========================

class LSTMSepsis(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 64, num_layers: int = 1, bidir: bool = False):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.bidir = bidir
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidir,
        )
        out_dim = hidden_dim * (2 if bidir else 1)
        self.classifier = nn.Sequential(
            nn.Linear(out_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, t, x):
        B, L, D = x.shape
        h0 = torch.zeros(self.num_layers * (2 if self.bidir else 1), B, self.hidden_dim, device=x.device)
        c0 = torch.zeros_like(h0)
        out, (hn, cn) = self.lstm(x, (h0, c0))
        last = out[:, -1, :]
        logits = self.classifier(last).squeeze(-1)
        return logits


model_lstm = LSTMSepsis(input_dim=input_dim, hidden_dim=64, num_layers=1, bidir=False).to(DEVICE)
optimizer_lstm = torch.optim.Adam(model_lstm.parameters(), lr=1e-3)

for epoch in range(1, epochs + 1):
    train_one_epoch(model_lstm, train_loader, optimizer_lstm, epoch, prefix="LSTM")
    evaluate(model_lstm, val_loader, prefix="LSTM")


[LSTM] Epoch 1 train loss = 0.1110
[LSTM] Val loss = 0.1205 | AUROC = 0.8092
[LSTM] Epoch 2 train loss = 0.0847
[LSTM] Val loss = 0.1488 | AUROC = 0.7724
[LSTM] Epoch 3 train loss = 0.0637
[LSTM] Val loss = 0.1894 | AUROC = 0.7478
[LSTM] Epoch 4 train loss = 0.0499
[LSTM] Val loss = 0.2223 | AUROC = 0.7285
[LSTM] Epoch 5 train loss = 0.0401
[LSTM] Val loss = 0.2590 | AUROC = 0.7090
[LSTM] Epoch 6 train loss = 0.0335
[LSTM] Val loss = 0.2634 | AUROC = 0.7152
[LSTM] Epoch 7 train loss = 0.0292
[LSTM] Val loss = 0.2932 | AUROC = 0.7035
[LSTM] Epoch 8 train loss = 0.0264
[LSTM] Val loss = 0.3012 | AUROC = 0.7285
[LSTM] Epoch 9 train loss = 0.0234
[LSTM] Val loss = 0.3166 | AUROC = 0.7174
[LSTM] Epoch 10 train loss = 0.0212
[LSTM] Val loss = 0.3277 | AUROC = 0.7248


In [8]:
# =========================
# 6. SAVE TRAINED MODELS
# =========================

os.makedirs("./models", exist_ok=True)

ode_save_path = "./models/neural_ode_sepsis.pt"
lstm_save_path = "./models/lstm_sepsis.pt"

# Convert means/stds to plain dicts for portability
means_dict = {k: float(v) for k, v in means.to_dict().items()}
stds_dict  = {k: float(v) for k, v in stds.to_dict().items()}

torch.save({
    "model_state_dict": model_ode.state_dict(),
    "input_dim": input_dim,
    "latent_dim": 32,
    "feature_cols": feature_cols,
    "means": means_dict,
    "stds": stds_dict,
    "patient_col": PATIENT_COL,
    "time_col": TIME_COL,
    "label_col": LABEL_COL,
}, ode_save_path)

torch.save({
    "model_state_dict": model_lstm.state_dict(),
    "input_dim": input_dim,
    "hidden_dim": 64,
    "num_layers": 1,
    "bidir": False,
    "feature_cols": feature_cols,
    "means": means_dict,
    "stds": stds_dict,
    "patient_col": PATIENT_COL,
    "time_col": TIME_COL,
    "label_col": LABEL_COL,
}, lstm_save_path)

print("Saved Neural ODE model to", ode_save_path)
print("Saved LSTM model to", lstm_save_path)


Saved Neural ODE model to ./models/neural_ode_sepsis.pt
Saved LSTM model to ./models/lstm_sepsis.pt


In [9]:
# =========================
# 7. EXTRA EVALUATION (AUPRC, CALIBRATION, THRESHOLDS)
# =========================

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    precision_recall_fscore_support,
    confusion_matrix,
)

@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    all_logits = []
    all_labels = []
    for t, x, y in loader:
        logits = model(t, x)
        all_logits.append(logits.cpu())
        all_labels.append(y.cpu())
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    probs = torch.sigmoid(all_logits).numpy()
    labels_np = all_labels.numpy()
    return probs, labels_np

def evaluate_rich(model, loader, name="Model", thresholds=(0.2, 0.5)):
    probs, labels_np = collect_predictions(model, loader)

    # AUROC already computed earlier; here we add AUPRC and calibration
    auroc = roc_auc_score(labels_np, probs)
    auprc = average_precision_score(labels_np, probs)
    brier = brier_score_loss(labels_np, probs)

    print(f"[{name}] AUROC = {auroc:.4f}, AUPRC = {auprc:.4f}, Brier = {brier:.4f}")

    for thr in thresholds:
        preds = (probs >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(labels_np, preds).ravel()
        precision, recall, f1, _ = precision_recall_fscore_support(labels_np, preds, average="binary", zero_division=0)
        specificity = tn / (tn + fp + 1e-8)
        print(f"  Threshold {thr:.2f}:")
        print(f"    TP={tp}, FP={fp}, FN={fn}, TN={tn}")
        print(f"    Sensitivity (Recall) = {recall:.3f}, Specificity = {specificity:.3f}, Precision = {precision:.3f}, F1 = {f1:.3f}")

    return {
        "auroc": auroc,
        "auprc": auprc,
        "brier": brier,
    }

# Run rich evaluation on validation set for both models
print("=== Rich evaluation on validation set ===")
metrics_ode  = evaluate_rich(model_ode,  val_loader, name="NeuralODE", thresholds=(0.1, 0.2, 0.3, 0.5))
metrics_lstm = evaluate_rich(model_lstm, val_loader, name="LSTM",     thresholds=(0.1, 0.2, 0.3, 0.5))

=== Rich evaluation on validation set ===
[NeuralODE] AUROC = 0.7448, AUPRC = 0.0760, Brier = 0.0402
  Threshold 0.10:
    TP=924, FP=8616, FN=1842, TN=87695
    Sensitivity (Recall) = 0.334, Specificity = 0.911, Precision = 0.097, F1 = 0.150
  Threshold 0.20:
    TP=631, FP=5630, FN=2135, TN=90681
    Sensitivity (Recall) = 0.228, Specificity = 0.942, Precision = 0.101, F1 = 0.140
  Threshold 0.30:
    TP=457, FP=4157, FN=2309, TN=92154
    Sensitivity (Recall) = 0.165, Specificity = 0.957, Precision = 0.099, F1 = 0.124
  Threshold 0.50:
    TP=272, FP=2367, FN=2494, TN=93944
    Sensitivity (Recall) = 0.098, Specificity = 0.975, Precision = 0.103, F1 = 0.101
[LSTM] AUROC = 0.7248, AUPRC = 0.0766, Brier = 0.0489
  Threshold 0.10:
    TP=626, FP=5077, FN=2140, TN=91234
    Sensitivity (Recall) = 0.226, Specificity = 0.947, Precision = 0.110, F1 = 0.148
  Threshold 0.20:
    TP=540, FP=4197, FN=2226, TN=92114
    Sensitivity (Recall) = 0.195, Specificity = 0.956, Precision = 0.114, F1 =

In [10]:
# =========================
# 8. INFERENCE HELPERS FOR NEURAL ODE
# =========================

def load_neural_ode_checkpoint(
    ckpt_path: str = "./models/neural_ode_sepsis.pt",
    map_location: str | torch.device = DEVICE,
):
    """
    Load Neural ODE checkpoint with all necessary metadata.
    """
    ckpt = torch.load(ckpt_path, map_location=map_location)

    input_dim = ckpt["input_dim"]
    latent_dim = ckpt["latent_dim"]
    feature_cols_ckpt = ckpt["feature_cols"]
    means_ckpt = ckpt["means"]
    stds_ckpt  = ckpt["stds"]
    patient_col_ckpt = ckpt["patient_col"]
    time_col_ckpt    = ckpt["time_col"]
    label_col_ckpt   = ckpt["label_col"]

    # Rebuild model
    model = NeuralODESepsis(input_dim=input_dim, latent_dim=latent_dim).to(map_location)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    # Convert means/stds back to Series-like structure
    means_series = pd.Series(means_ckpt)
    stds_series  = pd.Series(stds_ckpt)

    meta = {
        "feature_cols": feature_cols_ckpt,
        "means": means_series,
        "stds": stds_series,
        "patient_col": patient_col_ckpt,
        "time_col": time_col_ckpt,
        "label_col": label_col_ckpt,
    }

    return model, meta


def build_window_from_patient_df(
    df_patient: pd.DataFrame,
    meta: dict,
    window_size: int = 24,
):
    """
    Given a single patient's time-series DataFrame, build the last window for inference.
    Assumes df_patient already filtered to one patient and contains time + feature columns.
    """
    time_col = meta["time_col"]
    feature_cols = meta["feature_cols"]
    means = meta["means"]
    stds  = meta["stds"]

    df_p = df_patient.sort_values(time_col).reset_index(drop=True)

    # Normalize features using saved means/stds
    x = df_p[feature_cols].copy()
    x = (x - means[feature_cols]) / stds[feature_cols].replace(0, 1.0)

    times = df_p[time_col].values.astype(np.float32)
    feats = x.values.astype(np.float32)

    # Take last window_size rows (pad from beginning if shorter)
    if len(df_p) >= window_size:
        t_window = times[-window_size:]
        x_window = feats[-window_size:]
    else:
        pad = window_size - len(df_p)
        t_window = np.pad(times, (pad, 0), mode="edge")
        x_window = np.pad(feats, ((pad, 0), (0, 0)), mode="constant")

    # Normalize time to start at 0
    t_window = t_window - t_window[0]

    # Shape them for model: [B, L, D]
    t_tensor = torch.from_numpy(t_window).unsqueeze(0).to(DEVICE)       # [1, L]
    x_tensor = torch.from_numpy(x_window).unsqueeze(0).to(DEVICE)       # [1, L, D]

    return t_tensor, x_tensor


@torch.no_grad()
def predict_sepsis_risk_neural_ode(
    df_patient: pd.DataFrame,
    ckpt_path: str = "./models/neural_ode_sepsis.pt",
    window_size: int = 24,
) -> float:
    """
    High-level helper:
    - load model + meta
    - build last window from a single patient's DataFrame
    - return predicted sepsis risk in next horizon (probability between 0 and 1)
    """
    model, meta = load_neural_ode_checkpoint(ckpt_path=ckpt_path, map_location=DEVICE)
    t_tensor, x_tensor = build_window_from_patient_df(df_patient, meta, window_size=window_size)
    logits = model(t_tensor, x_tensor)  # [1]
    prob = torch.sigmoid(logits).item()
    return prob


# Example usage on a random validation patient
# (This is just a sanity check snippet; you can adapt it later.)

# Pick one patient from df_val and run inference
example_pid = df_val[PATIENT_COL].iloc[0]
df_example = df_val[df_val[PATIENT_COL] == example_pid]

print("Example patient ID:", example_pid, "| hours in record:", len(df_example))

risk = predict_sepsis_risk_neural_ode(df_example)
print("Predicted sepsis risk (next", horizon, "hours) for this patient:", f"{risk:.4f}")

Example patient ID: 1 | hours in record: 54
Predicted sepsis risk (next 6 hours) for this patient: 0.0000


In [11]:
# =========================
# A. CLASS WEIGHTS FOR IMBALANCE
# =========================

import numpy as np
import torch.nn.functional as F

# Estimate positive/negative fraction from training windows
all_labels_train = []

for _, _, y in train_ds:
    all_labels_train.append(float(y))

all_labels_train = np.array(all_labels_train)
pos_frac = all_labels_train.mean()
neg_frac = 1.0 - pos_frac

print(f"Training window labels: pos_frac = {pos_frac:.4f}, neg_frac = {neg_frac:.4f}")

# pos_weight for BCEWithLogitsLoss is neg/pos
if pos_frac > 0:
    pos_weight_value = neg_frac / pos_frac
else:
    pos_weight_value = 1.0  # fallback, should not happen

print("pos_weight for loss =", pos_weight_value)

pos_weight_tensor = torch.tensor(pos_weight_value, dtype=torch.float32, device=DEVICE)

criterion_weighted = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

Training window labels: pos_frac = 0.0302, neg_frac = 0.9698
pos_weight for loss = 32.11033691286171


In [12]:
# =========================
# B. IMPROVED NEURAL ODE MODEL (GRU ENCODER + RESIDUAL)
# =========================

class ODEFuncV2(nn.Module):
    def __init__(self, latent_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.Softplus(),
            nn.Linear(64, 64),
            nn.Softplus(),
            nn.Linear(64, latent_dim),
        )
        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, t, z):
        return self.net(z)


class NeuralODESepsisV2(nn.Module):
    """
    Improved version:
    - GRU encoder over the whole window to produce z0.
    - ODE evolution on z0.
    - Residual connection from last-step features into classifier.
    """
    def __init__(self, input_dim: int, latent_dim: int = 64, gru_hidden: int = 64, num_layers: int = 1):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=gru_hidden,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
        )

        # Map GRU hidden state to latent state for ODE
        self.to_latent = nn.Linear(gru_hidden, latent_dim)

        self.odefunc = ODEFuncV2(latent_dim)

        # Classifier sees both ODE latent and last-step features
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim + input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, t: torch.Tensor, x: torch.Tensor):
        """
        t: [B, L] (unused inside, but kept for interface consistency)
        x: [B, L, D]
        """
        B, L, D = x.shape

        # GRU encoder over the whole window
        # initial hidden state defaults to zeros
        out, h_n = self.gru(x)             # out: [B, L, H], h_n: [1, B, H]
        h_last = h_n[-1]                   # [B, H]

        # Map to latent for ODE
        z0 = self.to_latent(h_last)        # [B, latent_dim]

        # Integrate ODE over scaled [0, 1]
        t_eval = torch.tensor([0.0, 1.0], device=x.device)
        z_traj = odeint(self.odefunc, z0, t_eval)  # [T, B, latent_dim]
        zT = z_traj[-1]                              # [B, latent_dim]

        # Residual connection: concat ODE latent with last-step features
        x_last = x[:, -1, :]                        # [B, D]
        concat = torch.cat([zT, x_last], dim=-1)    # [B, latent_dim + D]

        logits = self.classifier(concat).squeeze(-1)
        return logits

In [13]:
# =========================
# C. TRAIN IMPROVED NEURAL ODE WITH WEIGHTED LOSS + SAVE
# =========================

def train_one_epoch_weighted(model, loader, optimizer, epoch: int, prefix: str = "Model"):
    model.train()
    total_loss = 0.0
    total = 0
    for t, x, y in loader:
        optimizer.zero_grad()
        logits = model(t, x)
        loss = criterion_weighted(logits, y)   # weighted loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * y.size(0)
        total += y.size(0)
    print(f"[{prefix}] Epoch {epoch} train loss = {total_loss/total:.4f}")


# Instantiate and train the improved model
input_dim = len(feature_cols)
model_ode_v2 = NeuralODESepsisV2(
    input_dim=input_dim,
    latent_dim=64,
    gru_hidden=64,
    num_layers=1,
).to(DEVICE)
optimizer_ode_v2 = torch.optim.Adam(model_ode_v2.parameters(), lr=1e-3)

epochs_v2 = 10  # you can increase later if it behaves well

for epoch in range(1, epochs_v2 + 1):
    train_one_epoch_weighted(model_ode_v2, train_loader, optimizer_ode_v2, epoch, prefix="NeuralODE_v2")
    evaluate(model_ode_v2, val_loader, prefix="NeuralODE_v2")   # uses existing evaluate() for AUROC


# Optional: rich evaluation (AUPRC/calibration) if you already defined evaluate_rich
print("=== Rich evaluation for improved Neural ODE (v2) on validation set ===")
metrics_ode_v2 = evaluate_rich(
    model_ode_v2,
    val_loader,
    name="NeuralODE_v2",
    thresholds=(0.1, 0.2, 0.3, 0.5),
)

# -------------------------
# Save improved model + meta
# -------------------------

os.makedirs("./models", exist_ok=True)
ode_v2_save_path = "./models/neural_ode_sepsis_v2.pt"

# means/stds from preprocessing step (convert to plain dicts)
means_dict = {k: float(v) for k, v in means.to_dict().items()}
stds_dict  = {k: float(v) for k, v in stds.to_dict().items()}

torch.save(
    {
        "model_state_dict": model_ode_v2.state_dict(),
        "input_dim": input_dim,
        "latent_dim": 64,
        "gru_hidden": 64,
        "num_layers": 1,
        "feature_cols": feature_cols,
        "means": means_dict,
        "stds": stds_dict,
        "patient_col": PATIENT_COL,
        "time_col": TIME_COL,
        "label_col": LABEL_COL,
        "window_size": window_size,
        "horizon": horizon,
        "class_pos_weight": float(pos_weight_tensor.item()),
    },
    ode_v2_save_path,
)

print("Saved improved Neural ODE v2 model to", ode_v2_save_path)

[NeuralODE_v2] Epoch 1 train loss = 0.9538
[NeuralODE_v2] Val loss = 0.5213 | AUROC = 0.7903
[NeuralODE_v2] Epoch 2 train loss = 0.6741
[NeuralODE_v2] Val loss = 0.3687 | AUROC = 0.7809
[NeuralODE_v2] Epoch 3 train loss = 0.4819
[NeuralODE_v2] Val loss = 0.3902 | AUROC = 0.7594
[NeuralODE_v2] Epoch 4 train loss = 0.4051
[NeuralODE_v2] Val loss = 0.3332 | AUROC = 0.7603
[NeuralODE_v2] Epoch 5 train loss = 0.3326
[NeuralODE_v2] Val loss = 0.4355 | AUROC = 0.7386
[NeuralODE_v2] Epoch 6 train loss = 0.2984
[NeuralODE_v2] Val loss = 0.4298 | AUROC = 0.7505
[NeuralODE_v2] Epoch 7 train loss = 0.2661
[NeuralODE_v2] Val loss = 0.3933 | AUROC = 0.7434
[NeuralODE_v2] Epoch 8 train loss = 0.2450
[NeuralODE_v2] Val loss = 0.3896 | AUROC = 0.7315
[NeuralODE_v2] Epoch 9 train loss = 0.2218
[NeuralODE_v2] Val loss = 0.4120 | AUROC = 0.7458
[NeuralODE_v2] Epoch 10 train loss = 0.2109
[NeuralODE_v2] Val loss = 0.4233 | AUROC = 0.7251
=== Rich evaluation for improved Neural ODE (v2) on validation set ==

## As the v2 is less performing we will stick to v1